# 1.0 - Preprocesamiento de datos

**MITSUI & CO. Commodity Prediction Challenge** (Kaggle).

Objetivo del notebook: explorar y preprocesar las series de precios de commodities
(`train.csv`, 557 features) y sus etiquetas (`train_labels.csv`, 424 targets), y dejar
listos `X.parquet` / `y.parquet` en `data/processed/`.

Las rutas se resuelven con `src/config.py` (lee `RAW_DATA_DIR` desde `.env`).

In [ ]:
import sys
from pathlib import Path

# Hacer importable el paquete src/ desde notebooks/
ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from src import config
from src.data.make_dataset import load_raw, preprocess, build_dataset

config.RAW_DATA_DIR, config.RAW_DATA_DIR.exists()

## 1) Cargar datos crudos

In [ ]:
features, labels, pairs = load_raw()
print('features:', features.shape, '| labels:', labels.shape, '| pairs:', pairs.shape)
features.head()

In [ ]:
# Mapa de targets: cada target es un instrumento o un spread (A - B) a cierto lag
pairs.head()

## 2) Exploración rápida

In [ ]:
# Valores faltantes por feature (instrumentos que no cotizan ciertos días)
na_ratio = features.drop(columns=[config.ID_COL]).isna().mean().sort_values(ascending=False)
na_ratio.head(10)

In [ ]:
# Distribución de lags de los targets
pairs['lag'].value_counts().sort_index()

## 3) Preprocesamiento y guardado

`preprocess()` ordena por `date_id` e imputa faltantes con ffill/bfill.
`build_dataset()` persiste `X.parquet` / `y.parquet` en `data/processed/`.

In [ ]:
X = preprocess(features)
print('NaN restantes en X:', int(X.drop(columns=[config.ID_COL]).isna().sum().sum()))
X.head()

In [ ]:
x_path, y_path = build_dataset()
print('Guardado:', x_path, y_path)